# Capa Silver — Limpieza y enriquecimiento

Toma los datos crudos de Bronze, los limpia, tipa y enriquece con joins a las tablas maestras.

**Prerequisito:** haber ejecutado el notebook `01_Bronze_Ingesta`.

In [0]:
# Recuperar el secreto de forma segura
storage_key = dbutils.secrets.get(scope="akv-scope", key="storage-key-value")

# Validás que no venga vacío
if not storage_key:
    raise ValueError("El parámetro storage_key no fue provisto.")

## Paso 1 — Configurar acceso al Storage Account

In [0]:
# Acceso directo al storage (sin mount, compatible con clusters Serverless)
storage_account = "icarostorage" # Reemplazar con tu storage
container       = "tpf-medallion-data" #Crear contenedor para medallion
key             = storage_key

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.blob.core.windows.net",
    key
)

BASE = f"wasbs://{container}@{storage_account}.blob.core.windows.net"

# Verificar que los archivos son accesibles
dbutils.fs.ls(BASE)

## Paso 2 — Leer las tablas Bronze

Los datos siguen siendo todo `string`, igual que los dejamos en Bronze. La conversión de tipos ocurre en el paso siguiente.

In [0]:
from pyspark.sql.functions import to_date

# Leer las cuatro tablas desde Bronze
pokemon    = spark.read.format("delta").load(f"{BASE}/pokemon")
pokemones_metadata    = spark.read.format("delta").load(f"{BASE}/pokemones_metadata")

print("Filas en pokemon (raw):", pokemon.count())
print("Filas en metadata (raw):", pokemones_metadata.count())

## Paso 3 — Limpieza: deduplicación y conversión de fecha

Eliminamos duplicados usando las columnas clave del negocio y convertimos `fecha` al tipo `DateType` con formato explícito para evitar errores de parseo.

In [0]:
from pyspark.sql import functions as F
# Aplanar results y sacar el ID de la URL
pokemones_api_flat = (
    pokemon
    .select(F.explode("results").alias("item"))
    .select(
        F.col("item.name").alias("api_name"),
        F.regexp_extract("item.url", r"/pokemon/(\d+)/", 1).cast("int").alias("pokemon_id"),
        F.col("item.url").alias("api_url")
    )
)

print("Pokemones Silver procesados:", pokemones_api_flat.count(), "filas")
display(pokemones_api_flat)

In [0]:
# Deduplicar por clave de negocio y tipar la fecha
pokemon_clean = pokemones_api_flat \
    .dropDuplicates(["pokemon_id", "api_url", "api_name"]) \
    .withColumnRenamed("api_name", "pokemon_name")

print("Filas después de limpieza:", pokemon_clean.count())
pokemon_clean.printSchema()

## Paso 4 — Enriquecimiento con joins a tablas maestras

Hacemos `inner join` contra clientes, productos y tiendas. El resultado es una sola tabla con toda la información necesaria para análisis.

> **Nota:** `Clientes.csv` tiene una columna `ciudad` extra — la dropeamos antes del join para evitar ambigüedad con la columna `ciudad` de tiendas.

In [0]:
# Dropear columna ciudad de clientes para evitar conflicto con ciudad de tiendas
#clientes = clientes.drop("ciudad")

# Join de ventas con las tres tablas maestras
pokemon_silver = pokemon_clean \
    .join(pokemones_metadata,  on="pokemon_id",  how="inner")

print("Filas en tabla Silver:", pokemon_silver.count())
pokemon_silver.printSchema()


## Paso 5 — Guardar la tabla Silver en Delta

In [0]:
# Persistir la tabla Silver en el blob storage
pokemon_silver.write.format("delta"). \
    mode("overwrite").option("overwriteSchema", "true").save(f"{BASE}/silver/pokemon_silver")

print("Tabla Silver guardada en:", f"{BASE}/silver/pokemon_silver")


## Paso 6 — Verificación

In [0]:
# Verificar la tabla Silver
silver_check = spark.read.format("delta").load(f"{BASE}/silver/pokemon_silver")
silver_check.show(5)
